In [1]:
import pandas as pd

In [ ]:
import sys
import re
import time
import json
from pathlib import Path
from tqdm import tqdm
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


class EnglishToRomanizedNepali:
    def __init__(self, model_name: str = "mistral-small3.1:latest", temperature: float = 0.0, batch_size: int = 16):
        self.model_name = model_name
        self.temperature = temperature
        self.batch_size = batch_size
        self.llm = None
        self._initialize_model()

    def _initialize_model(self):
        try:
            self.llm = ChatOllama(
                model=self.model_name,
                temperature=self.temperature,
                num_predict=2048,
                num_ctx=4096,
            )
            print(f"✓ Model '{self.model_name}' loaded", file=sys.stderr)
            print(f"✓ Batch size: {self.batch_size} lines per API call", file=sys.stderr)
        except Exception as e:
            print(f"✗ Failed to load model: {e}", file=sys.stderr)
            sys.exit(1)

    def _system_prompt(self):
        return """You are Nepali by nationality and prefer converting english into romanized-Nepali, have the expert level understanding of word level conversion,You know where to use chha and chh,
        also when to convert the english word into romanized nepali, also convert the date, for example may-4 becomes Baishakh-22"""
        if not texts:
            return texts
        
        # Filter out empty lines but remember their positions
        non_empty_indices = [i for i, t in enumerate(texts) if t.strip()]
        empty_indices = [i for i, t in enumerate(texts) if not t.strip()]
        
        if not non_empty_indices:
            return texts  # All lines are empty
        
        # Prepare batch of non-empty texts
        batch_texts = [texts[i] for i in non_empty_indices]
        
        # Join with a unique separator that won't appear in the text
        separator = "\n###\n"
        batch_input = separator.join(batch_texts)
        
        prompt = ChatPromptTemplate.from_messages([
            ("system", self._system_prompt() + f"\n\nConvert each of the following {len(batch_texts)} lines to romanized Nepali. Separate each output with '{separator.strip()}' on a new line. Keep the exact same order. No extra text or numbering."),
            ("human", "{text}")
        ])
        
        try:
            chain = prompt | self.llm | StrOutputParser()
            result = chain.invoke({"text": batch_input}).strip()
            
            # Split back into individual results
            batch_results = [r.strip() for r in result.split(separator)]
            
            # Validate we got the right number of results
            if len(batch_results) != len(batch_texts):
                print(f"  ⚠ Batch size mismatch: expected {len(batch_texts)}, got {len(batch_results)}. Falling back to individual processing.", file=sys.stderr)
                # Fallback to individual processing for this batch
                batch_results = [self.romanize_single(text) for text in batch_texts]
            
            # Clean each result (remove any Devanagari)
            batch_results = [re.sub(r'[\u0900-\u097F]+', '', r).strip() or t for r, t in zip(batch_results, batch_texts)]
            
            # Reconstruct full results with empty lines in original positions
            full_results = [""] * len(texts)
            for idx, result_text in zip(non_empty_indices, batch_results):
                full_results[idx] = result_text
            for idx in empty_indices:
                full_results[idx] = texts[idx]  # Preserve empty lines
            
            return full_results
            
        except Exception as e:
            print(f"  ✗ Batch error: {e}. Falling back to individual processing.", file=sys.stderr)
            # Fallback to individual processing
            return [self.romanize_single(text) for text in texts]

    def romanize_single(self, text: str) -> str:
        """Fallback method for processing one line at a time"""
        if not text.strip():
            return text
        try:
            prompt = ChatPromptTemplate.from_messages([
                ("system", self._system_prompt() + "\n\nConvert this single line to romanized Nepali. Output ONLY the romanized text, no explanation."),
                ("human", "{text}")
            ])
            chain = prompt | self.llm | StrOutputParser()
            result = chain.invoke({"text": text}).strip()
            # Strip any accidental Devanagari
            result = re.sub(r'[\u0900-\u097F]+', '', result).strip()
            return result if result else text
        except Exception as e:
            print(f"  ✗ Error on line: {e}", file=sys.stderr)
            return text

    def process_file(self, input_path: Path, output_path: Path, resume: bool = True):
        start_time = time.time()
        print(f"\n📄 Processing: {input_path.name}", file=sys.stderr)

        # Read input file
        with open(input_path, 'r', encoding='utf-8') as f:
            lines = [line.rstrip('\n') for line in f]

        total = len(lines)
        print(f"   Total lines: {total:,}", file=sys.stderr)

        # Handle resume functionality
        processed_count = 0
        results = []

        if resume and output_path.exists():
            with open(output_path, 'r', encoding='utf-8') as f:
                results = [line.rstrip('\n') for line in f]
                processed_count = len(results)
                if processed_count < total:
                    print(f"   Resuming from line {processed_count:,} ({processed_count/total*100:.1f}% complete)", file=sys.stderr)
                    lines = lines[processed_count:]
                    results = results[:processed_count]  # Keep already processed results
                else:
                    print(f"   ✓ Already complete, skipping.", file=sys.stderr)
                    return

        if not lines:
            print(f"   ✓ Already complete, skipping.", file=sys.stderr)
            return

        # Process in batches
        total_batches = (len(lines) + self.batch_size - 1) // self.batch_size
        print(f"   Processing {len(lines):,} lines in {total_batches} batches (batch size: {self.batch_size})", file=sys.stderr)
        
        with tqdm(
            total=len(lines),
            desc=f"  {input_path.name[:30]}",
            unit="lines",
            bar_format='{l_bar}{bar:35}{r_bar}'
        ) as pbar:
            
            for batch_start in range(0, len(lines), self.batch_size):
                batch_end = min(batch_start + self.batch_size, len(lines))
                batch_lines = lines[batch_start:batch_end]
                
                # Process the batch
                batch_results = self.romanize_batch(batch_lines)
                
                # Add to results
                results.extend(batch_results)
                
                # Update progress
                pbar.update(len(batch_lines))
                
                # Save checkpoint every few batches (after every 50 lines or so)
                if len(results) % 50 < self.batch_size:  # Save periodically
                    self._save_results(results, output_path)
        
        # Final save
        self._save_results(results, output_path)

        elapsed = time.time() - start_time
        speed = total / elapsed if elapsed > 0 else 0
        print(f"   ✅ Done in {self._format_time(elapsed)} ({speed:.1f} lines/sec) -> {output_path}", file=sys.stderr)

    def _save_results(self, results: list, output_path: Path):
        """Save results to file"""
        output_path.parent.mkdir(parents=True, exist_ok=True)
        # Use temporary file to avoid corruption
        temp_path = output_path.with_suffix('.tmp')
        with open(temp_path, 'w', encoding='utf-8') as f:
            for line in results:
                f.write(line + '\n')
        # Rename for atomic operation
        temp_path.replace(output_path)

    def _format_time(self, seconds: float) -> str:
        if seconds < 60:      return f"{seconds:.0f}s"
        elif seconds < 3600:  return f"{seconds/60:.1f}m"
        elif seconds < 86400: return f"{seconds/3600:.1f}h"
        else:                 return f"{seconds/86400:.1f}d"


def main():
    # ── Configuration ────────────────────────────────────────────────────────
    SOURCE_FOLDER  = "economy"
    OUTPUT_FOLDER  = "economy-romanized"
    MODEL_NAME     = "mistral-small3.1:latest"
    BATCH_SIZE     = 10      # Process 10 lines per API call (adjust based on your needs)
    TEMPERATURE    = 0.0
    FILE_EXTENSION = ".txt"
    RESUME         = True
    # ─────────────────────────────────────────────────────────────────────────

    print("=" * 70, file=sys.stderr)
    print("  ENGLISH -> ROMANIZED NEPALI — FOLDER BATCH PROCESSOR", file=sys.stderr)
    print("=" * 70, file=sys.stderr)

    source_path = Path(SOURCE_FOLDER)
    output_path = Path(OUTPUT_FOLDER)

    if not source_path.exists() or not source_path.is_dir():
        print(f"✗ Error: Source folder '{SOURCE_FOLDER}' not found.", file=sys.stderr)
        sys.exit(1)

    input_files = sorted(source_path.glob(f"*{FILE_EXTENSION}"))
    if not input_files:
        print(f"✗ No {FILE_EXTENSION} files found in '{SOURCE_FOLDER}'.", file=sys.stderr)
        sys.exit(1)

    print(f"\n✓ Source  : {source_path.resolve()}", file=sys.stderr)
    print(f"✓ Output  : {output_path.resolve()}", file=sys.stderr)
    print(f"✓ Files   : {len(input_files)}", file=sys.stderr)
    print(f"✓ Model   : {MODEL_NAME}", file=sys.stderr)
    print(f"✓ Batch Size: {BATCH_SIZE} lines per API call", file=sys.stderr)

    print(f"\nFiles:", file=sys.stderr)
    for f in input_files:
        size_kb = f.stat().st_size / 1024
        out_f = output_path / f.name
        status = "✓ done" if (RESUME and out_f.exists()) else "-> pending"
        print(f"  {status}  {f.name}  ({size_kb:.1f} KB)", file=sys.stderr)

    romanizer = EnglishToRomanizedNepali(MODEL_NAME, TEMPERATURE, BATCH_SIZE)
    output_path.mkdir(parents=True, exist_ok=True)

    grand_start = time.time()
    completed = 0

    for idx, in_file in enumerate(input_files, 1):
        out_file = output_path / in_file.name

        # Skip completely processed files
        if RESUME and out_file.exists():
            with open(in_file, 'r', encoding='utf-8') as fi:
                in_count = sum(1 for _ in fi)
            with open(out_file, 'r', encoding='utf-8') as fo:
                out_count = sum(1 for _ in fo)
            if in_count == out_count:
                print(f"\n[{idx}/{len(input_files)}] ⏭  Skipping (complete): {in_file.name}", file=sys.stderr)
                completed += 1
                continue

        print(f"\n[{idx}/{len(input_files)}] Processing: {in_file.name}", file=sys.stderr)
        romanizer.process_file(in_file, out_file, resume=RESUME)
        completed += 1

    grand_elapsed = time.time() - grand_start
    print(f"\n{'=' * 70}", file=sys.stderr)
    print(f"  ALL DONE!", file=sys.stderr)
    print(f"  Files processed: {completed}/{len(input_files)}", file=sys.stderr)
    print(f"  Total time: {romanizer._format_time(grand_elapsed)}", file=sys.stderr)
    print(f"  Output folder: {output_path.resolve()}", file=sys.stderr)
    print(f"{'=' * 70}", file=sys.stderr)


if __name__ == "__main__":
    main()

  ENGLISH -> ROMANIZED NEPALI — FOLDER BATCH PROCESSOR

✓ Source  : /home/lang-chain/Documents/tiny_LLM_scratch_with_tokenizer/onliine_news_nepal_english/economy
✓ Output  : /home/lang-chain/Documents/tiny_LLM_scratch_with_tokenizer/onliine_news_nepal_english/economy-romanized
✓ Files   : 400
✓ Model   : mistral-small3.1:latest
✓ Batch Size: 10 lines per API call

Files:
  -> pending  article_0001.txt  (2.1 KB)
  -> pending  article_0002.txt  (1.0 KB)
  -> pending  article_0003.txt  (1.0 KB)
  -> pending  article_0004.txt  (1.6 KB)
  -> pending  article_0005.txt  (2.0 KB)
  -> pending  article_0006.txt  (1.5 KB)
  -> pending  article_0007.txt  (2.0 KB)
  -> pending  article_0008.txt  (1.0 KB)
  -> pending  article_0009.txt  (4.6 KB)
  -> pending  article_0010.txt  (1.6 KB)
  -> pending  article_0011.txt  (1.0 KB)
  -> pending  article_0012.txt  (1.1 KB)
  -> pending  article_0013.txt  (1.0 KB)
  -> pending  article_0014.txt  (3.3 KB)
  -> pending  article_0015.txt  (8.5 KB)
  -> pendin

  article_0001.txt: 100%|███████████████████████████████████| 29/29 [03:06<00:00,  6.41s/lines]
   ✅ Done in 3.1m (0.2 lines/sec) -> economy-romanized/article_0001.txt

[2/400] Processing: article_0002.txt

📄 Processing: article_0002.txt
   Total lines: 19
   Processing 19 lines in 2 batches (batch size: 10)
  article_0002.txt: 100%|███████████████████████████████████| 19/19 [01:33<00:00,  4.92s/lines]
   ✅ Done in 1.6m (0.2 lines/sec) -> economy-romanized/article_0002.txt

[3/400] Processing: article_0003.txt

📄 Processing: article_0003.txt
   Total lines: 19
   Processing 19 lines in 2 batches (batch size: 10)
  article_0003.txt: 100%|███████████████████████████████████| 19/19 [01:36<00:00,  5.10s/lines]
   ✅ Done in 1.6m (0.2 lines/sec) -> economy-romanized/article_0003.txt

[4/400] Processing: article_0004.txt

📄 Processing: article_0004.txt
   Total lines: 21
   Processing 21 lines in 3 batches (batch size: 10)
  article_0004.txt:   0%|                                   | 0/21 [00

In [1]:
import ntr

# Convert a Devanagari sentence to romanized Nepali
romanized_text = ntr.nep_to_rom("म नेपाल मा बस्छु ।")
print(romanized_text)
# Output: ma nepal ma baschhu .

m nepaal maa baschhu .
